# Normal toy example using SBI

In [1]:
import torch
import numpy as np
from sbi.inference import NPE_C, simulate_for_sbi
from sbi.utils.user_input_checks import (
    check_sbi_inputs,
    process_prior,
    process_simulator,
)
from sbi.utils.torchutils import BoxUniform
from scipy.stats import norm, uniform
from sbi.analysis import plot_summary
import matplotlib.pyplot as plt
import seaborn as sns


torch_device = "cpu"
torch.set_default_device(torch_device)

c:\Users\u2008181\likelihood-free\sbi_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\u2008181\likelihood-free\sbi_env\Lib\site-packages\arviz\__init__.py:39: FutureWarning: 
ArviZ is undergoing a major refactor to improve flexibility and extensibility while maintaining a user-friendly interface.
Some upcoming changes may be backward incompatible.
For details and migration guidance, visit: https://python.arviz.org/en/latest/user_guide/migration_guide.html
  warn(
WARNING (pytensor.configdefaults): g++ not available, if using conda: `conda install gxx`
WARNING (pytensor.configdefaults): g++ not detected!  PyTensor will be unable to compile C-implementations and will default to Python. Performance may be severely degraded. To remove this warning, set PyTensor flags cxx to an empty string.


## Model

**Prior:**
-  $\mu \sim \text{Uniform}[0, 10]$
-  $n \sim \text{Discrete Uniform}[10, 200]$

**Simulator:**
-  $x_{1:n} \sim N(\mu, 1)$
-  $s_1 = (\overline{x}, n)$
-  $s_2 = (\overline{x}, \overline{x^2}, n)$

In [5]:
prior_torch = BoxUniform(
    low=torch.tensor([0.0, 10.0], device=torch_device),
    high=torch.tensor([10.0, 200.0], device=torch_device),
    device=torch_device
)

In [3]:
def simulator_torch_1(theta):
    x = torch.randn(theta[1].int(), device=torch_device) + theta[0]
    s1 = torch.tensor([torch.mean(x), theta[1]], device=torch_device)
    return s1

In [4]:
def simulator_torch_2(theta):
    x = torch.randn(theta[1].int(), device=torch_device) + theta[0]
    s2 = torch.tensor([torch.mean(x), torch.mean(x**2), theta[1]], device=torch_device)
    return s2

Set the true values to be:
-  $\mu = 3$,
-  $n = 50$.

The observed summary statistics are shown below as

In [6]:
torch.manual_seed(1)
np.random.seed(1)
true_theta = torch.tensor([3.0, 50.0], device=torch_device)
x_o_2 = simulator_torch_2(true_theta)
x_o_1 = x_o_2[[0, 2]]
true_theta, x_o_1, x_o_2

(tensor([ 3., 50.]),
 tensor([ 2.8323, 50.0000]),
 tensor([ 2.8323,  9.0894, 50.0000]))

## NPE-C

In [7]:
simulation_budget = 50000
seed = 1
prior = prior_torch
num_epochs=500
num_posterior_samples=1000

In [8]:
simulator = simulator_torch_2

prior, num_parameters, prior_returns_numpy = process_prior(prior)
simulator = process_simulator(simulator, prior, prior_returns_numpy)
check_sbi_inputs(simulator, prior)

inference = NPE_C(prior=prior, density_estimator="nsf", device=torch_device)

learning_rate = 0.0005  # default value

torch.manual_seed(seed)
np.random.seed(seed)

theta, x = simulate_for_sbi(
    simulator=simulator, proposal=prior, num_simulations=simulation_budget
)

100%|██████████| 50000/50000 [00:02<00:00, 19066.01it/s]


In [11]:
x_1 = x[:, [0, 2]]
x_2 = x
x_1, x_2

(tensor([[1.6835e-01, 2.1846e+01],
         [2.2213e+00, 1.2969e+02],
         [8.4909e+00, 1.1891e+02],
         ...,
         [4.0948e+00, 1.0975e+02],
         [5.1318e+00, 1.0424e+02],
         [4.9051e+00, 1.7177e+02]]),
 tensor([[1.6835e-01, 9.7783e-01, 2.1846e+01],
         [2.2213e+00, 5.9477e+00, 1.2969e+02],
         [8.4909e+00, 7.3189e+01, 1.1891e+02],
         ...,
         [4.0948e+00, 1.7745e+01, 1.0975e+02],
         [5.1318e+00, 2.7433e+01, 1.0424e+02],
         [4.9051e+00, 2.4892e+01, 1.7177e+02]]))

**Simulator 1**

In [ ]:
torch.manual_seed(seed)
np.random.seed(seed)

density_estimator = inference.append_simulations(theta, x_1).train(
    max_num_epochs=num_epochs, 
    learning_rate=learning_rate,
    stop_after_epochs=20
)

fig, axes = plot_summary(
    inference, 
    tags=["training_loss", "validation_loss"], 
    figsize=(8, 4)
)
plt.title("Training and Validation Loss")
plt.show()

 Training neural network. Epochs trained: 94

In [ ]:
posterior_1 = inference.build_posterior(density_estimator)

Plug in observed data to get the posterior.

In [ ]:
theta_trained = posterior_1.set_default_x(x_o_1).sample((num_posterior_samples,), x=x_o_1)
theta_trained = theta_trained.reshape((num_posterior_samples, 2))
theta_trained_numpy = theta_trained.cpu().numpy()

**Simulator 2**

In [ ]:
torch.manual_seed(seed)
np.random.seed(seed)

density_estimator = inference.append_simulations(theta, x_2).train(
    max_num_epochs=num_epochs, 
    learning_rate=learning_rate,
    stop_after_epochs=20
)

fig, axes = plot_summary(
    inference, 
    tags=["training_loss", "validation_loss"], 
    figsize=(8, 4)
)
plt.title("Training and Validation Loss")
plt.show()

In [ ]:
posterior_2 = inference.build_posterior(density_estimator)

Plug in observed data to get the posterior.

In [ ]:
theta_trained = posterior_2.set_default_x(x_o_2).sample((num_posterior_samples,), x=x_o_2)
theta_trained = theta_trained.reshape((num_posterior_samples, 2))
theta_trained_numpy = theta_trained.cpu().numpy()

**True posterior by the truncated normal distribution**

In [ ]:
prior_a = 0.0
prior_b = 10.0
theta_grid = np.linspace(0, 10, 1000)
x_o_2_numpy = x_o_2.cpu().numpy()

prior = uniform.pdf(theta_grid, loc=prior_a, scale=(prior_b - prior_a))

likelihood = norm.pdf(x_o_2_numpy[0], loc=theta_grid, scale=1/np.sqrt(x_o_2_numpy[1]))

unnormalized_posterior = likelihood * prior
d_theta = theta_grid[1] - theta_grid[0]
posterior = unnormalized_posterior / (np.sum(unnormalized_posterior) * d_theta)

In [ ]:
plt.figure(figsize=(12, 8))
sns.set_style('whitegrid')
sns.kdeplot(posterior_1[:, 0], label='Posterior 1', color='blue', linewidth=1, alpha=0.5)
sns.kdeplot(posterior_2[:, 0], label='Posterior 2', color='orange', linewidth=1, alpha=0.5)
plt.plot(theta_grid, posterior, label='True Posterior', color='green', linewidth=2, alpha=0.5)
plt.legend()
plt.show()

## Change number of simulations

500 simulations

In [ ]:
posterior_1_500 = norm_example_npe_c(500, 1, prior_torch, x_o_1, simulator_torch_1)

In [ ]:
posterior_2_500 = norm_example_npe_c(500, 1, prior_torch, x_o_2, simulator_torch_2)

In [ ]:
posterior_3_500 = norm_example_npe_c(500, 1, prior_torch, x_o_3, simulator_torch_3)

In [ ]:
posterior_4_500 = norm_example_npe_c(500, 1, prior_torch, x_o_4, simulator_torch_4)

In [ ]:
plt.figure(figsize=(12, 8))
sns.set_style('whitegrid')
sns.kdeplot(posterior_1_500[:, 0], label='Posterior 1', color='blue', linewidth=1, alpha=0.5)
sns.kdeplot(posterior_2_500[:, 0], label='Posterior 2', color='orange', linewidth=1, alpha=0.5)
sns.kdeplot(posterior_3_500[:, 0], label='Posterior 3', color='red', linewidth=1, alpha=0.5)
sns.kdeplot(posterior_4_500[:, 0], label='Posterior 4', color='purple', linewidth=1, alpha=0.5)
plt.plot(theta_grid, posterior, label='True Posterior', color='green', linewidth=2, alpha=0.5)
plt.legend()
plt.show()

10000 simulations

In [ ]:
posterior_1_10000 = norm_example_npe_c(10000, 1, prior_torch, x_o_1, simulator_torch_1)

In [ ]:
posterior_2_10000 = norm_example_npe_c(10000, 1, prior_torch, x_o_2, simulator_torch_2)

In [ ]:
posterior_3_10000 = norm_example_npe_c(10000, 1, prior_torch, x_o_3, simulator_torch_3)

In [ ]:
posterior_4_10000 = norm_example_npe_c(10000, 1, prior_torch, x_o_4, simulator_torch_4)

In [ ]:
plt.figure(figsize=(12, 8))
sns.set_style('whitegrid')
sns.kdeplot(posterior_1_10000[:, 0], label='Posterior 1', color='blue', linewidth=1, alpha=0.5)
sns.kdeplot(posterior_2_10000[:, 0], label='Posterior 2', color='orange', linewidth=1, alpha=0.5)
sns.kdeplot(posterior_3_10000[:, 0], label='Posterior 3', color='red', linewidth=1, alpha=0.5)
sns.kdeplot(posterior_4_10000[:, 0], label='Posterior 4', color='purple', linewidth=1, alpha=0.5)
plt.plot(theta_grid, posterior, label='True Posterior', color='green', linewidth=2, alpha=0.5)
plt.legend()
plt.show()

50000 simulations

In [ ]:
posterior_1_50000 = norm_example_npe_c(50000, 1, prior_torch, x_o_1, simulator_torch_1)

In [ ]:
posterior_2_50000 = norm_example_npe_c(50000, 1, prior_torch, x_o_2, simulator_torch_2)

In [ ]:
posterior_3_50000 = norm_example_npe_c(50000, 1, prior_torch, x_o_3, simulator_torch_3)

In [ ]:
posterior_4_50000 = norm_example_npe_c(50000, 1, prior_torch, x_o_4, simulator_torch_4)

In [ ]:
plt.figure(figsize=(12, 8))
sns.set_style('whitegrid')
sns.kdeplot(posterior_1_50000[:, 0], label='Posterior 1', color='blue', linewidth=1, alpha=0.5)
sns.kdeplot(posterior_2_50000[:, 0], label='Posterior 2', color='orange', linewidth=1, alpha=0.5)
sns.kdeplot(posterior_3_50000[:, 0], label='Posterior 3', color='red', linewidth=1, alpha=0.5)
sns.kdeplot(posterior_4_50000[:, 0], label='Posterior 4', color='purple', linewidth=1, alpha=0.5)
plt.plot(theta_grid, posterior, label='True Posterior', color='green', linewidth=2, alpha=0.5)
plt.legend()
plt.show()

Change our true parameter as $\mu=5, n=20$

In [ ]:
torch.manual_seed(1)
np.random.seed(1)
true_theta = torch.tensor([5.0, 20.0], device=torch_device)
x_o_4 = simulator_torch_4(true_theta)
x_o_1 = x_o_4[0]
x_o_2 = x_o_4[[0, 2]]
x_o_3 = x_o_4[[0, 1]]
true_theta, x_o_1, x_o_2, x_o_3, x_o_4

In [ ]:
posterior_2_mu5 = norm_example_npe_c(50000, 1, prior_torch, x_o_2, simulator_torch_2)

In [ ]:
posterior_4_mu5 = norm_example_npe_c(50000, 1, prior_torch, x_o_4, simulator_torch_4)

In [ ]:
prior_a = 0.0
prior_b = 10.0
theta_grid = np.linspace(0, 10, 1000)
x_o_2_numpy = x_o_2.cpu().numpy()

prior = uniform.pdf(theta_grid, loc=prior_a, scale=(prior_b - prior_a))

likelihood = norm.pdf(x_o_2_numpy[0], loc=theta_grid, scale=1/np.sqrt(x_o_2_numpy[1]))

unnormalized_posterior = likelihood * prior
d_theta = theta_grid[1] - theta_grid[0]
posterior = unnormalized_posterior / (np.sum(unnormalized_posterior) * d_theta)

In [ ]:
plt.figure(figsize=(12, 8))
sns.set_style('whitegrid')
sns.kdeplot(posterior_2_mu5[:, 0], label='Posterior 2', color='orange', linewidth=1, alpha=0.5)
sns.kdeplot(posterior_4_mu5[:, 0], label='Posterior 4', color='purple', linewidth=1, alpha=0.5)
plt.plot(theta_grid, posterior, label='True Posterior', color='green', linewidth=2, alpha=0.5)
plt.legend()
plt.show()

Change our true parameter as $\mu=6, n=500$

In [ ]:
torch.manual_seed(1)
np.random.seed(1)
true_theta = torch.tensor([6.0, 500.0], device=torch_device)
x_o_4 = simulator_torch_4(true_theta)
x_o_1 = x_o_4[0]
x_o_2 = x_o_4[[0, 2]]
x_o_3 = x_o_4[[0, 1]]
true_theta, x_o_1, x_o_2, x_o_3, x_o_4

In [ ]:
posterior_2_mu6 = norm_example_npe_c(50000, 1, prior_torch, x_o_2, simulator_torch_2)

In [ ]:
posterior_4_mu6 = norm_example_npe_c(50000, 1, prior_torch, x_o_4, simulator_torch_4)

In [ ]:
prior_a = 0.0
prior_b = 10.0
theta_grid = np.linspace(0, 10, 1000)
x_o_2_numpy = x_o_2.cpu().numpy()

prior = uniform.pdf(theta_grid, loc=prior_a, scale=(prior_b - prior_a))

likelihood = norm.pdf(x_o_2_numpy[0], loc=theta_grid, scale=1/np.sqrt(x_o_2_numpy[1]))

unnormalized_posterior = likelihood * prior
d_theta = theta_grid[1] - theta_grid[0]
posterior = unnormalized_posterior / (np.sum(unnormalized_posterior) * d_theta)

In [ ]:
plt.figure(figsize=(12, 8))
sns.set_style('whitegrid')
sns.kdeplot(posterior_2_mu6[:, 0], label='Posterior 2', color='orange', linewidth=1, alpha=0.5)
sns.kdeplot(posterior_4_mu6[:, 0], label='Posterior 4', color='purple', linewidth=1, alpha=0.5)
plt.plot(theta_grid, posterior, label='True Posterior', color='green', linewidth=2, alpha=0.5)
plt.legend()
plt.show()

NPE needs more simulations than I expected. This can also be shown in `normal_simple.ipynb`. But in real simulator, we can sample only for the largest $n$ and take subsets.

We can see that without including $n$, the posterior distributions have wrong variances. And without $E[x^2]$, NPE is unsensitive to different variance.

**Questions:** how many simulations do we need, when we don't know the true posterior? Neural networks identify convergence only by validation loss.

**Metrics:**

Simulation-based calibration (SBC) do diagnostics based on ranks. It is sensitive to over/under-confident posteriors.

Classifier two-sample test (C2ST) may also sensitive to over/under-confident posteriors (if we know the true posterior).

## NLE

In [ ]:
def norm_example_nle(
    simulation_budget, seed, prior, x_obs, simulator, num_epochs=500, num_posterior_samples=1000
):
    prior, num_parameters, prior_returns_numpy = process_prior(prior)
    simulator = process_simulator(simulator, prior, prior_returns_numpy)
    check_sbi_inputs(simulator, prior)

    # density_estimator_fun = likelihood_nn(
    #     model="maf",
    #     hidden_features=50,
    #     z_score_x="independent",
    #     z_score_theta="independent",
    # )

    inference = NLE_A(prior=prior, density_estimator='maf', device=torch_device)

    learning_rate = 0.0005

    torch.manual_seed(seed)
    np.random.seed(seed)

    theta, x = simulate_for_sbi(
        simulator=simulator, proposal=prior, num_simulations=simulation_budget
    )

    density_estimator = inference.append_simulations(theta, x).train(
        max_num_epochs=num_epochs, 
        learning_rate=learning_rate,
        stop_after_epochs=20
    )

    fig, axes = plot_summary(
        inference, 
        tags=["training_loss", "validation_loss"], 
        figsize=(8, 4)
    )
    plt.title("Training and Validation Loss")
    plt.show()

    posterior = inference.build_posterior(density_estimator).set_default_x(x_obs)

    theta_trained = posterior.sample((num_posterior_samples,), x=x_obs)
    theta_trained = theta_trained.reshape((num_posterior_samples, 2))

    return theta_trained.cpu().numpy()

In [ ]:
posterior_1 = norm_example_nle(50000, 1, prior_torch, x_o_1, simulator_torch_1)

In [ ]:
posterior_2 = norm_example_nle(50000, 1, prior_torch, x_o_2, simulator_torch_2)

In [ ]:
posterior_3 = norm_example_nle(50000, 1, prior_torch, x_o_3, simulator_torch_3)

In [ ]:
posterior_4 = norm_example_nle(50000, 1, prior_torch, x_o_4, simulator_torch_4)

In [ ]:
plt.figure(figsize=(12, 8))
sns.set_style('whitegrid')
# sns.kdeplot(posterior_1[:, 0], label='Posterior 1', color='blue', linewidth=1, alpha=0.5)
sns.kdeplot(posterior_2[:, 0], label='Posterior 2', color='orange', linewidth=1, alpha=0.5)
# sns.kdeplot(posterior_3[:, 0], label='Posterior 3', color='red', linewidth=1, alpha=0.5)
sns.kdeplot(posterior_4[:, 0], label='Posterior 4', color='purple', linewidth=1, alpha=0.5)
plt.plot(theta_grid, posterior, label='True Posterior', color='green', linewidth=2, alpha=0.5)
plt.legend()
plt.show()